In [ ]:
import numpy as np
import cv2
import os
import json
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)

print("="*70)
print("FIXED PREPROCESSING PIPELINE")
print("="*70)

## Configuration

In [ ]:
# ============================================================================
# CONFIGURATION - MODIFY ONLY THIS SECTION
# ============================================================================

import yaml

# Load config.yaml
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Paths from config
PROJECT_ROOT = Path(config['paths']['project_root'])
RAW_DATA_DIR = Path(config['paths']['raw_data'])

# Output directory
OUTPUT_DIR = Path(config['paths']['preprocessed_data'])

# Image parameters from config
IMG_SIZE = config['data']['image_size']
CLASS_NAMES = config['data']['class_names']

# Split ratios from config
TRAIN_RATIO = config['data']['split_ratios']['train']
VAL_RATIO = config['data']['split_ratios']['val']
# Remaining for internal test (from config: internal_test ratio)
# Official "Testing" folder used as final held-out test

# Preprocessing parameters from config
preprocess_config = config['data']['preprocessing']
BILATERAL_D = preprocess_config['bilateral_filter']['d']
BILATERAL_SIGMA_COLOR = preprocess_config['bilateral_filter']['sigma_color']
BILATERAL_SIGMA_SPACE = preprocess_config['bilateral_filter']['sigma_space']
COLORMAP = getattr(cv2, preprocess_config['colormap'])  # Convert string to cv2 constant

print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw Data: {RAW_DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Classes: {CLASS_NAMES}")

## Step 1: Validate Raw Data

In [ ]:
# Validate raw data exists
print("\n" + "="*70)
print("STEP 1: VALIDATING RAW DATA")
print("="*70)

train_dir = RAW_DATA_DIR / "Training"
test_dir = RAW_DATA_DIR / "Testing"

assert train_dir.exists(), f"Training directory not found: {train_dir}"
assert test_dir.exists(), f"Testing directory not found: {test_dir}"

# Count samples
train_counts = {}
test_counts = {}

for class_name in CLASS_NAMES:
    train_class_dir = train_dir / class_name
    test_class_dir = test_dir / class_name
    
    if train_class_dir.exists():
        train_files = list(train_class_dir.glob('*.jpg')) + list(train_class_dir.glob('*.png'))
        train_counts[class_name] = len(train_files)
    else:
        train_counts[class_name] = 0
        print(f"  ⚠️  Warning: No training data for {class_name}")
    
    if test_class_dir.exists():
        test_files = list(test_class_dir.glob('*.jpg')) + list(test_class_dir.glob('*.png'))
        test_counts[class_name] = len(test_files)
    else:
        test_counts[class_name] = 0
        print(f"  ⚠️  Warning: No testing data for {class_name}")

print("\nRaw Data Statistics:")
print("-" * 70)
print(f"{'Class':<15} {'Training':<12} {'Testing':<12} {'Total':<12}")
print("-" * 70)

total_train = 0
total_test = 0

for class_name in CLASS_NAMES:
    train_count = train_counts[class_name]
    test_count = test_counts[class_name]
    total = train_count + test_count
    
    print(f"{class_name:<15} {train_count:<12} {test_count:<12} {total:<12}")
    total_train += train_count
    total_test += test_count

print("-" * 70)
print(f"{'TOTAL':<15} {total_train:<12} {total_test:<12} {total_train + total_test:<12}")
print("="*70)

assert total_train > 0, "No training data found!"
assert total_test > 0, "No testing data found!"
print("\n✅ Raw data validation passed!")

## Step 2: Define Preprocessing Function

In [ ]:
def preprocess_image(img_path, target_size=(IMG_SIZE, IMG_SIZE)):
    """
    Apply consistent preprocessing to a single image.
    
    Steps:
    1. Read grayscale image
    2. Apply bilateral filter (edge-preserving denoising)
    3. Apply BONE colormap (medical imaging standard)
    4. Resize to target size
    5. Keep as uint8 (0-255) - normalization done during training
    
    Args:
        img_path: Path to input image
        target_size: (height, width) for output
    
    Returns:
        Preprocessed image as RGB numpy array (H, W, 3), uint8
    """
    # Read as grayscale
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        raise ValueError(f"Could not read image: {img_path}")
    
    # Bilateral filter - removes noise while preserving edges
    # Critical for medical images where tumor boundaries are important
    img_filtered = cv2.bilateralFilter(
        img, 
        d=BILATERAL_D, 
        sigmaColor=BILATERAL_SIGMA_COLOR, 
        sigmaSpace=BILATERAL_SIGMA_SPACE
    )
    
    # Apply BONE colormap - converts grayscale to pseudo-color
    # Bone colormap designed for medical imaging (X-ray/MRI)
    # Creates 3-channel image suitable for pretrained ImageNet models
    img_colored = cv2.applyColorMap(img_filtered, COLORMAP)
    
    # Resize to target size
    img_resized = cv2.resize(img_colored, target_size, interpolation=cv2.INTER_AREA)
    
    # Convert BGR to RGB (OpenCV uses BGR by default)
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    
    return img_rgb

# Test preprocessing on a sample image
print("\n" + "="*70)
print("TESTING PREPROCESSING FUNCTION")
print("="*70)

# Find a sample image
sample_class = CLASS_NAMES[0]
sample_dir = train_dir / sample_class
sample_images = list(sample_dir.glob('*.jpg'))

if not sample_images:
    sample_images = list(sample_dir.glob('*.png'))

if sample_images:
    sample_img_path = sample_images[0]
    sample_img = preprocess_image(sample_img_path)
    
    print(f"✅ Successfully preprocessed: {sample_img_path.name}")
    print(f"   Shape: {sample_img.shape}")
    print(f"   Dtype: {sample_img.dtype}")
    print(f"   Range: [{sample_img.min()}, {sample_img.max()}]")
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    # Original
    original = cv2.imread(str(sample_img_path), cv2.IMREAD_GRAYSCALE)
    axes[0].imshow(original, cmap='gray')
    axes[0].set_title('Original (Grayscale)')
    axes[0].axis('off')
    
    # Preprocessed
    axes[1].imshow(sample_img)
    axes[1].set_title(f'Preprocessed ({IMG_SIZE}x{IMG_SIZE}, BONE colormap)')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR.parent / 'preprocessing_sample.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Preprocessing function validated!")
else:
    print("⚠️  No sample images found for testing")

## Step 3: Create Train/Val Splits from Training Data

In [ ]:
print("\n" + "="*70)
print("STEP 3: CREATING TRAIN/VAL SPLITS")
print("="*70)

# Collect all training files with labels
all_train_files = []
all_train_labels = []

for class_idx, class_name in enumerate(CLASS_NAMES):
    class_dir = train_dir / class_name
    
    if class_dir.exists():
        # Get all image files
        files = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
        
        for file in files:
            all_train_files.append(file)
            all_train_labels.append(class_idx)

print(f"Total training samples: {len(all_train_files)}")

# First split: train+val (90%) vs internal_test (10%)
train_val_files, internal_test_files, train_val_labels, internal_test_labels = train_test_split(
    all_train_files,
    all_train_labels,
    test_size=0.10,
    stratify=all_train_labels,
    random_state=SEED
)

# Second split: train (80% of train+val) vs val (20% of train+val) = 72% vs 18% of original
# This gives approximately 72/18/10 split for train/val/internal_test
train_files, val_files, train_labels, val_labels = train_test_split(
    train_val_files,
    train_val_labels,
    test_size=0.20,  # 20% of 90% = 18% of total
    stratify=train_val_labels,
    random_state=SEED
)

print(f"\nSplit Statistics:")
print(f"  Training set:       {len(train_files):5d} samples ({len(train_files)/len(all_train_files)*100:.1f}%)")
print(f"  Validation set:     {len(val_files):5d} samples ({len(val_files)/len(all_train_files)*100:.1f}%)")
print(f"  Internal test set:  {len(internal_test_files):5d} samples ({len(internal_test_files)/len(all_train_files)*100:.1f}%)")

# Check class distribution
print("\nClass Distribution:")
print("-" * 70)
print(f"{'Class':<15} {'Train':<10} {'Val':<10} {'Test':<10}")
print("-" * 70)

for class_idx, class_name in enumerate(CLASS_NAMES):
    train_count = sum(1 for label in train_labels if label == class_idx)
    val_count = sum(1 for label in val_labels if label == class_idx)
    test_count = sum(1 for label in internal_test_labels if label == class_idx)
    
    print(f"{class_name:<15} {train_count:<10} {val_count:<10} {test_count:<10}")

print("="*70)
print("✅ Splits created with stratification (balanced class distribution)")

## Step 4: Preprocess and Save All Splits

In [ ]:
print("\n" + "="*70)
print("STEP 4: PREPROCESSING AND SAVING")
print("="*70)

# Create output directories
splits = {
    'train': (train_files, train_labels),
    'val': (val_files, val_labels),
    'internal_test': (internal_test_files, internal_test_labels)
}

# Also process the official Testing folder (held-out)
test_files = []
test_labels = []

for class_idx, class_name in enumerate(CLASS_NAMES):
    test_class_dir = test_dir / class_name
    if test_class_dir.exists():
        files = list(test_class_dir.glob('*.jpg')) + list(test_class_dir.glob('*.png'))
        for file in files:
            test_files.append(file)
            test_labels.append(class_idx)

splits['heldout_test'] = (test_files, test_labels)

# Process each split
split_info = {}
failed_images = []

for split_name, (files, labels) in splits.items():
    print(f"\nProcessing {split_name} set ({len(files)} images)...")
    
    # Create split directory
    split_dir = OUTPUT_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    
    # Create class subdirectories
    for class_name in CLASS_NAMES:
        (split_dir / class_name).mkdir(exist_ok=True)
    
    # Process images
    split_file_mapping = {}
    
    for img_path, label in tqdm(zip(files, labels), total=len(files), desc=f'  {split_name}'):
        try:
            # Preprocess
            processed_img = preprocess_image(img_path)
            
            # Save to appropriate class folder
            class_name = CLASS_NAMES[label]
            output_path = split_dir / class_name / img_path.name
            
            # Save as PNG for lossless storage
            # (will be loaded and normalized during training)
            cv2.imwrite(str(output_path), cv2.cvtColor(processed_img, cv2.COLOR_RGB2BGR))
            
            # Record mapping
            split_file_mapping[str(img_path)] = {
                'output_path': str(output_path),
                'class': class_name,
                'class_idx': int(label)
            }
            
        except Exception as e:
            print(f"\n  ⚠️  Failed to process {img_path.name}: {str(e)}")
            failed_images.append((str(img_path), str(e)))
    
    split_info[split_name] = {
        'num_images': len(files),
        'class_distribution': {class_name: sum(1 for l in labels if l == idx) 
                               for idx, class_name in enumerate(CLASS_NAMES)},
        'file_mapping': split_file_mapping
    }

print("\n" + "="*70)
print("PREPROCESSING SUMMARY")
print("="*70)

for split_name, info in split_info.items():
    print(f"\n{split_name.upper()}:")
    print(f"  Total images: {info['num_images']}")
    print(f"  Class distribution:")
    for class_name, count in info['class_distribution'].items():
        print(f"    {class_name}: {count}")

if failed_images:
    print(f"\n⚠️  Failed to process {len(failed_images)} images:")
    for img_path, error in failed_images:
        print(f"    {img_path}: {error}")
else:
    print("\n✅ All images processed successfully!")

## Step 5: Save Metadata and Generate Report

In [ ]:
print("\n" + "="*70)
print("STEP 5: SAVING METADATA")
print("="*70)

# Save split information as JSON
metadata = {
    'preprocessing_config': {
        'image_size': IMG_SIZE,
        'bilateral_filter': {
            'd': BILATERAL_D,
            'sigmaColor': BILATERAL_SIGMA_COLOR,
            'sigmaSpace': BILATERAL_SIGMA_SPACE
        },
        'colormap': 'COLORMAP_BONE',
        'seed': SEED
    },
    'splits': split_info,
    'class_names': CLASS_NAMES,
    'failed_images': failed_images
}

metadata_path = OUTPUT_DIR / 'preprocessing_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Metadata saved to: {metadata_path}")

# Generate text report
report_path = OUTPUT_DIR / 'preprocessing_report.txt'
with open(report_path, 'w') as f:
    f.write("="*70 + "\n")
    f.write("PREPROCESSING REPORT\n")
    f.write("="*70 + "\n\n")
    
    f.write("Configuration:\n")
    f.write(f"  Image Size: {IMG_SIZE}x{IMG_SIZE}\n")
    f.write(f"  Bilateral Filter: d={BILATERAL_D}, sigmaColor={BILATERAL_SIGMA_COLOR}, sigmaSpace={BILATERAL_SIGMA_SPACE}\n")
    f.write(f"  Colormap: COLORMAP_BONE\n")
    f.write(f"  Random Seed: {SEED}\n\n")
    
    f.write("Data Splits:\n")
    f.write("-"*70 + "\n")
    
    for split_name, info in split_info.items():
        f.write(f"\n{split_name.upper()}:\n")
        f.write(f"  Total: {info['num_images']} images\n")
        f.write(f"  Distribution:\n")
        for class_name, count in info['class_distribution'].items():
            percentage = (count / info['num_images']) * 100
            f.write(f"    {class_name}: {count} ({percentage:.1f}%)\n")
    
    if failed_images:
        f.write(f"\nFailed Images ({len(failed_images)}):")
        for img_path, error in failed_images:
            f.write(f"  {img_path}: {error}\n")
    
    f.write("\n" + "="*70 + "\n")
    f.write("Preprocessing completed successfully!\n")
    f.write("="*70 + "\n")

print(f"✅ Report saved to: {report_path}")

print("\n" + "="*70)
print("✅ PREPROCESSING COMPLETE!")
print("="*70)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nYou can now use this data for all experiments with:")
print(f"  - Train: {OUTPUT_DIR / 'train'}")
print(f"  - Validation: {OUTPUT_DIR / 'val'}")
print(f"  - Internal Test: {OUTPUT_DIR / 'internal_test'}")
print(f"  - Held-out Test: {OUTPUT_DIR / 'heldout_test'}")
print("\nAll experiments MUST use this preprocessed data for fair comparison!")

## Step 6: Validation Checks

In [ ]:
print("\n" + "="*70)
print("STEP 6: VALIDATION CHECKS")
print("="*70)

# Check 1: All splits exist
print("\nCheck 1: Directory structure")
for split_name in ['train', 'val', 'internal_test', 'heldout_test']:
    split_path = OUTPUT_DIR / split_name
    assert split_path.exists(), f"Missing directory: {split_path}"
    print(f"  ✅ {split_name}: {split_path}")

# Check 2: Class folders exist in each split
print("\nCheck 2: Class folders")
for split_name in ['train', 'val', 'internal_test', 'heldout_test']:
    for class_name in CLASS_NAMES:
        class_path = OUTPUT_DIR / split_name / class_name
        assert class_path.exists(), f"Missing class folder: {class_path}"
    print(f"  ✅ {split_name}: All class folders present")

# Check 3: No data leakage (no overlap between splits)
print("\nCheck 3: Data leakage detection")
train_stems = set(f.stem for f in train_files)
val_stems = set(f.stem for f in val_files)
test_stems = set(f.stem for f in internal_test_files)

train_val_overlap = train_stems & val_stems
train_test_overlap = train_stems & test_stems
val_test_overlap = val_stems & test_stems

assert len(train_val_overlap) == 0, f"Data leakage: {len(train_val_overlap)} files in both train and val"
assert len(train_test_overlap) == 0, f"Data leakage: {len(train_test_overlap)} files in both train and test"
assert len(val_test_overlap) == 0, f"Data leakage: {len(val_test_overlap)} files in both val and test"

print("  ✅ No data leakage detected (train/val/test are disjoint)")

# Check 4: Image properties
print("\nCheck 4: Image properties")
sample_path = list((OUTPUT_DIR / 'train' / CLASS_NAMES[0]).glob('*.png'))[0]
sample_img = cv2.imread(str(sample_path))
assert sample_img.shape == (IMG_SIZE, IMG_SIZE, 3), f"Wrong image shape: {sample_img.shape}"
assert sample_img.dtype == np.uint8, f"Wrong dtype: {sample_img.dtype}"
print(f"  ✅ Image shape: {sample_img.shape}")
print(f"  ✅ Image dtype: {sample_img.dtype}")
print(f"  ✅ Image range: [{sample_img.min()}, {sample_img.max()}]")

# Check 5: Metadata exists
print("\nCheck 5: Metadata files")
assert metadata_path.exists(), "Metadata JSON missing"
assert report_path.exists(), "Report text file missing"
print(f"  ✅ Metadata: {metadata_path}")
print(f"  ✅ Report: {report_path}")

print("\n" + "="*70)
print("✅ ALL VALIDATION CHECKS PASSED!")
print("="*70)
print("\n🎉 Preprocessing pipeline completed successfully!")
print("\nNext steps:")
print("1. Review preprocessing_report.txt for statistics")
print("2. Use preprocessed_canonical/ directory for ALL experiments")
print("3. NEVER modify this preprocessed data")
print("4. Re-run this notebook if you need to change preprocessing")